In [1]:
import random
from pathlib import Path
from collections import namedtuple
import torch
from torch.nn.utils.rnn import pad_packed_sequence, pack_padded_sequence

from datasets import load_dataset
import torch.nn as nn

import tokenizers
import logging


In [2]:

# ----------------------------------
# Logging configuration
# ----------------------------------

LOG_FORMAT = (
    "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

formatter = logging.Formatter(LOG_FORMAT)

# Console: INFO and above
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

# File: DEBUG and above
file_handler = logging.FileHandler(
    "training.log",
    mode="a",
    encoding="utf-8"
)

file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

# Configure root logger
logging.basicConfig(
    level=logging.DEBUG,
    handlers=[
        console_handler,
        file_handler
    ],
    force=True
)


# ----------------------------------
# Enable / Disable logging
# ----------------------------------

def enable_logging():
    """Enable all logging levels."""
    logging.disable(logging.NOTSET)


def disable_logging():
    """Disable all logging levels."""
    logging.disable(logging.CRITICAL)


# Enable logging by default
# enable_logging()
disable_logging()

In [3]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

In [4]:
def get_data():
    for data in nmt_train_set:
        yield data["source_text"], data["target_text"]

    for data in nmt_test_set:
        yield data["source_text"], data["target_text"]

tokenizer_path = Path("tokenziers") / "NMT.json"
tokenizer = tokenizers.Tokenizer(tokenizers.models.BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.ByteLevel()
tokenizer.decoder = tokenizers.decoders.ByteLevel()

if tokenizer_path.exists():
    logging.info(f"Found existing tokenzier. Loading from {tokenizer_path}!")
    tokenizer = tokenizers.Tokenizer.from_file(str(tokenizer_path))
else:
    logging.info(f"not found existing tokenzier. saving to {tokenizer_path}!")
    token_trainer = tokenizers.trainers.BpeTrainer(
        vocab_size=1000, special_tokens=["[PAD]", "[UNK]", "[MASK]", "[CLS]", "[SEP]", "[SOS]", "[EOS]"]
    )

    tokenizer.train_from_iterator(get_data(),token_trainer)

    tokenizer.save(str(tokenizer_path))

tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")

print("Vocab size:", tokenizer.get_vocab_size())
print("PAD ID:", tokenizer.token_to_id("[PAD]"))
print("UNK ID:", tokenizer.token_to_id("[UNK]"))

encoded = tokenizer.encode('Tom tried to break up the fight.')

print("Tokens:", encoded.tokens)
print("IDs:", encoded.ids)
print("Decoded:", tokenizer.decode(encoded.ids))

Vocab size: 1000
PAD ID: 0
UNK ID: 1
Tokens: ['ĠTom', 'Ġtr', 'i', 'ed', 'Ġto', 'Ġb', 're', 'ak', 'Ġup', 'Ġthe', 'Ġf', 'ight', '.']
IDs: [211, 353, 75, 202, 204, 196, 193, 464, 540, 209, 200, 447, 20]
Decoded:  Tom tried to break up the fight.


In [35]:
def attention(query, key, value):  # note: dq == dk and Lk == Lv
    scores = query @ key.transpose(1, 2)  # [B,Lq,dq] @ [B,dk,Lk] = [B, Lq, Lk]
    weights = torch.softmax(scores, dim=-1)  # [B, Lq, Lk]
    return weights @ value
    
class NMTRNN(nn.Module):
    def __init__(self, vocab_size:int,hidden_size:int = 128, gru_num_layers:int = 1, embedding_dim:int=128, padding_idx:int = 0):
        super().__init__()
        
        self.embedding_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_dim,padding_idx=padding_idx)

        self.encoder = nn.GRU(
            input_size=embedding_dim, 
            hidden_size=hidden_size, 
            num_layers = gru_num_layers, 
            batch_first=True
        )

        self.decoder = nn.GRU(
            input_size=hidden_size, 
            hidden_size=hidden_size, 
            num_layers = gru_num_layers, 
            batch_first=True
        )

        # multiplied by 2 , cuz we are going to concat the attention output to the 
        # decoder output which will be of same size and hence double the input size is not required
        self.classifier = nn.Linear(in_features=hidden_size *2, out_features=vocab_size)


    def forward(self, X):
    
        # print(f"X: {X.src_ids.shape}")
    
        input_embeddings = self.embedding_layer(X.src_ids)
        # print(f"input_embeddings: {input_embeddings.shape}")
    
        src_packed = pack_padded_sequence(
            input_embeddings,
            X.src_mask.sum(dim=1).to(device="cpu"),
            batch_first=True,
            enforce_sorted=False
        )
    
        encoder_outputs_packed, hidden_state = self.encoder(src_packed)
        # print(f"encoder_output hidden_state: {hidden_state.shape}")
    
        encoder_outputs, _ = pad_packed_sequence(
            encoder_outputs_packed,
            batch_first=True
        )
    
        output_embeddings = self.embedding_layer(X.dst_ids)
        # print(f"output_embeddings: {output_embeddings.shape}")
    
        decoder_output, _ = self.decoder(output_embeddings, hidden_state)
        # print(f"decoder_output: {decoder_output.shape}")
    
        attn_output = attention(
            query=decoder_output,
            key=encoder_outputs,
            value=encoder_outputs
        )
        # print(f"attn_output: {attn_output.shape}")
    
        output = self.classifier(torch.cat((attn_output, decoder_output), dim=-1))
        # print(f"output: {output.shape}")
    
        output = output.transpose(1, 2)
        # print(f"permutted output: {output.shape}")
    
        return output


class_items = ["src_ids", "src_mask", "dst_ids", "dst_mask"]

class NMTPair(namedtuple("NMTPairBase", class_items)):
    def to(self, device:torch.device):
        return NMTPair(
            self.src_ids.to(device=device), 
            self.src_mask.to(device=device),
            self.dst_ids.to(device=device), 
            self.dst_mask.to(device=device)
        )


def collat_fn(batch):
    source_texts = [f"[SOS]{item['source_text']}" for item in batch]
    dst_texts = [f"[SOS]{item['target_text']}[EOS]" for item in batch]

    encoded_source = tokenizer.encode_batch(source_texts)
    encoded_dst = tokenizer.encode_batch(dst_texts)

    source_ids = torch.tensor([item.ids for item in encoded_source])
    source_masks = torch.tensor([item.attention_mask for item in encoded_source])
    
    dst_ids = torch.tensor([item.ids for item in encoded_dst])
    dst_masks = torch.tensor([item.attention_mask for item in encoded_dst])
    
    data = NMTPair(source_ids, source_masks, dst_ids[:, :-1], dst_masks[:, :-1])

    return data, dst_ids[ : , 1:]
    

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = NMTRNN(
    vocab_size=tokenizer.get_vocab_size(),
    hidden_size = 128, 
    gru_num_layers = 1, 
    embedding_dim=128, 
    padding_idx = tokenizer.token_to_id("[PAD]")
).to(device=device)

In [56]:
BATCH_SIZE = 64
PIN_MEM = True
NUM_PROCESSES = 0

train_loader = torch.utils.data.DataLoader(
    nmt_train_set, 
    shuffle=True, 
    num_workers=NUM_PROCESSES, 
    batch_size=BATCH_SIZE, 
    pin_memory=PIN_MEM,
    collate_fn=collat_fn    
)

test_loader = torch.utils.data.DataLoader(
    nmt_valid_set, 
    num_workers=NUM_PROCESSES, 
    batch_size=BATCH_SIZE, 
    pin_memory=PIN_MEM,
    collate_fn=collat_fn
)


EPOCHS = 50
LOG_ITEM_AT = 1000
LR = 5e-5
loss_fn = torch.nn.CrossEntropyLoss()
optim = torch.optim.AdamW(params=model.parameters(), lr=LR)


In [57]:
len(train_loader)

2467

In [58]:

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0
    test_loss = 0
    for idx, (x, y) in enumerate(train_loader, start=1):
        x = x.to(device=device)
        y = y.to(device=device, dtype=torch.long)
        output = model(x)
        loss = loss_fn(output, y)
        del x
        train_loss += loss.item()
        optim.zero_grad()
        loss.backward()
        optim.step()
        if idx == 1 or idx == len(train_loader) or idx % LOG_ITEM_AT == 0:
            logging.info(f"[TRAIN] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(train_loader):05d}] | [Train: {train_loss/idx:5f}]")
            print(f"[TRAIN] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(train_loader):05d}] | [Train: {train_loss/idx:5f}]")
            
    model.eval()
    with torch.inference_mode():
        for idx, (x, y) in enumerate(test_loader, start=1):
            x = x.to(device=device)
            y = y.to(device=device, dtype=torch.long)
            output = model(x)
            loss = loss_fn(output, y)
            del x
            test_loss += loss.item()
            if idx == 1 or idx == len(test_loader) or idx % LOG_ITEM_AT == 0:
                logging.info(f"[TEST] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(test_loader):05d}] | [Test: {test_loss/idx:.5f}]")
                print(f"[TEST] | [EPOCH: {epoch:04d}] | [STEP: {idx:05d} / {len(test_loader):05d}] | [Test: {test_loss/idx:.5f}]")


    logging.info(f"[EPOCH: {epoch:04d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/len(test_loader):.5f}]")
    print(f"[EPOCH: {epoch:04d}] | [Train: {train_loss/len(train_loader):5f}] | [Test: {test_loss/len(test_loader):.5f}]")
        

[TRAIN] | [EPOCH: 0001] | [STEP: 00001 / 02467] | [Train: 0.136758]
[TRAIN] | [EPOCH: 0001] | [STEP: 01000 / 02467] | [Train: 0.295491]
[TRAIN] | [EPOCH: 0001] | [STEP: 02000 / 02467] | [Train: 0.293090]
[TRAIN] | [EPOCH: 0001] | [STEP: 02467 / 02467] | [Train: 0.294586]
[TEST] | [EPOCH: 0001] | [STEP: 00001 / 00617] | [Test: 0.38487]
[TEST] | [EPOCH: 0001] | [STEP: 00617 / 00617] | [Test: 0.39300]
[EPOCH: 0001] | [Train: 0.294586] | [Test: 0.39300]
[TRAIN] | [EPOCH: 0002] | [STEP: 00001 / 02467] | [Train: 0.327014]
[TRAIN] | [EPOCH: 0002] | [STEP: 01000 / 02467] | [Train: 0.292664]
[TRAIN] | [EPOCH: 0002] | [STEP: 02000 / 02467] | [Train: 0.293055]
[TRAIN] | [EPOCH: 0002] | [STEP: 02467 / 02467] | [Train: 0.293304]
[TEST] | [EPOCH: 0002] | [STEP: 00001 / 00617] | [Test: 0.38447]
[TEST] | [EPOCH: 0002] | [STEP: 00617 / 00617] | [Test: 0.39300]
[EPOCH: 0002] | [Train: 0.293304] | [Test: 0.39300]
[TRAIN] | [EPOCH: 0003] | [STEP: 00001 / 02467] | [Train: 0.312812]
[TRAIN] | [EPOCH: 0003] 

KeyboardInterrupt: 

In [59]:
nmt_model_path = Path("models") / "NMT_with_attn.pt"

nmt_model_path.parent.mkdir(parents=True, exist_ok=True)

In [60]:
torch.save(model.state_dict(), str(nmt_model_path))
# model.load_state_dict(torch.load(str(nmt_model_path)))

In [52]:
def translate(sample: dict, max_length: int = 100):

    was_training = model.training
    model.eval()

    # Special token IDs
    sos_id = tokenizer.token_to_id("[SOS]")
    eos_id = tokenizer.token_to_id("[EOS]")

    # --------------------------------------------------
    # 1. Encode source sentence
    # Match the collator's source preprocessing
    # --------------------------------------------------

    source_text = f"[SOS] {sample['source_text']} [EOS]"

    encoded_source = tokenizer.encode(source_text)

    source_ids = torch.tensor(
        encoded_source.ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    source_mask = torch.tensor(
        encoded_source.attention_mask,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)

    # --------------------------------------------------
    # 2. Initialize decoder input with SOS
    # --------------------------------------------------

    decoded_ids = torch.tensor(
        [[sos_id]],
        dtype=torch.long,
        device=device
    )

    generated_ids = []

    # --------------------------------------------------
    # 3. Autoregressive generation
    # --------------------------------------------------

    try:
        with torch.inference_mode():

            for step in range(max_length):

                # Decoder mask matches decoder input length
                decoded_mask = torch.ones_like(decoded_ids)

                # Same structure as your collator
                data = NMTPair(
                    source_ids,
                    source_mask,
                    decoded_ids,
                    decoded_mask
                )

                # Forward pass
                logits = model(data)

                # Determine output layout
                vocab_size = tokenizer.get_vocab_size()

                next_token_logits = logits[:, :, -1]

                # Greedy decoding
                next_token = torch.argmax(
                    next_token_logits,
                    dim=-1,
                    keepdim=True
                )

                token_id = next_token.item()

                # Stop when EOS is generated
                if token_id == eos_id:
                    print(f"[EOS] generated at step {step + 1}")
                    break

                generated_ids.append(token_id)

                # Append prediction to decoder input
                decoded_ids = torch.cat(
                    [decoded_ids, next_token],
                    dim=1
                )

    finally:
        if was_training:
            model.train()

    # --------------------------------------------------
    # 4. Decode generated sequence
    # --------------------------------------------------

    translation = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    print("Source:     ", sample["source_text"])

    if "target_text" in sample:
        print("Reference:  ", sample["target_text"])

    print("Translation:", translation)

    return translation

In [53]:

idx = random.randrange(len(test_loader.dataset))

sample = test_loader.dataset[idx]

translation = translate(sample)

[EOS] generated at step 33
Source:      Through which city does the Mississippi River pass?
Reference:   ¿Por qué ciudad pasa el río Mississippi?
Translation:  Trá qué ciudad la ciudad de ciudad la Missibiana pasa la vista?


In [13]:
sample

{'source_text': 'Can we see it now?',
 'target_text': '¿Podemos verlo ahora?',
 'source_lang': 'eng',
 'target_lang': 'spa'}